# Notebook 01 — Setup & Field Semantics

**Goal**: Connect to WorldQuant Brain, fetch all data fields for each dataset, 
generate LLM-powered semantic descriptions, and store them in the Chroma vector store.

Run this notebook **once** before anything else. Results are persisted to disk.

## Prerequisites
```bash
pip install -e ".[dev]"
cp .env.example .env   # fill in WQB_USERNAME, WQB_PASSWORD, LLM_MODEL, OPENAI_API_KEY
```

In [1]:
import asyncio
import nest_asyncio

nest_asyncio.apply()  # allow asyncio in Jupyter

from alpha_agent.config import settings

settings.ensure_dirs()

print("LLM model:", settings.llm_model)
print("Chroma dir:", settings.chroma_persist_dir)
print("DuckDB path:", settings.duckdb_path)
print("WQB username:", settings.wqb_username or "⚠ NOT SET")

LLM model: deepseek/deepseek-reasoner
Chroma dir: data/chroma
DuckDB path: data/alpha_memory.db
WQB username: freedom8625@163.com


## 1. Initialize the vector store

In [2]:
from alpha_agent.knowledge.vector_store import VectorStore

store = VectorStore()
print("Vector store initialized at:", settings.chroma_persist_dir)

/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/myenv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Vector store initialized at: data/chroma


## 2. Connect to WQB and fetch data fields

In [3]:
from alpha_agent.data.wqb_client import WQBClient
from alpha_agent.data.datafield_indexer import DataFieldIndexer

DATASETS = [
    {"id": "fundamental6", "universe": "TOP3000"},
    {"id": "analyst4",     "universe": "TOP1000"},
    {"id": "pv1",          "universe": "TOP1000"},
]

async def index_all_datasets():
    async with WQBClient() as client:
        indexer = DataFieldIndexer(client, store)
        all_enriched = {}
        for ds in DATASETS:
            print(f"\n=== Indexing dataset: {ds['id']} ===")
            enriched = await indexer.index_dataset(
                dataset_id=ds["id"],
                universe=ds["universe"],
            )
            all_enriched[ds["id"]] = enriched
            print(f"  Indexed {len(enriched)} fields")
        return all_enriched

all_fields = asyncio.run(index_all_datasets())


=== Indexing dataset: fundamental6 ===
[DataFieldIndexer] Fetching fields for dataset=fundamental6 universe=TOP3000
[DataFieldIndexer] Got 48 fields, enriching with LLM semantics...

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.set_verbose=True'.


Provider List: https://docs.litellm.ai/docs/providers

[DataFieldIndexer] LLM error for debt_lt: APIError: DeepseekException - deepseek does not support parameters: {'response_format': {'type': 'json_object'}}, for model=deepseek-reasoner. To drop these, set `litellm.drop_params=True` or for proxy:

`litellm_settings:
 drop_params: true`


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.set_verbose=True'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this 

## 3. Explore the indexed field semantics

In [4]:
# Count fields indexed per dataset
from alpha_agent.knowledge.vector_store import COLLECTION_FIELDS

total = store.count(COLLECTION_FIELDS)
print(f"Total fields indexed in vector store: {total}")

for ds_id, fields in all_fields.items():
    print(f"  {ds_id}: {len(fields)} fields")

Total fields indexed in vector store: 169
  fundamental6: 48 fields
  analyst4: 27 fields
  pv1: 12 fields


In [5]:
# Semantic search test
query = "earnings quality and profitability"
results = store.query(COLLECTION_FIELDS, query, k=5)

print(f"Top 5 fields for query: '{query}'\n")
for r in results:
    print(f"  [{r['metadata'].get('field_id')}] dist={r['distance']:.3f}")
    print(f"  {r['document'][:150]}\n")

Top 5 fields for query: 'earnings quality and profitability'

  [actual_eps_value_quarterly] dist=0.547
  actual_eps_value_quarterly

  [actual_sales_value_quarterly] dist=0.575
  actual_sales_value_quarterly

  [actual_dividend_value_quarterly] dist=0.595
  actual_dividend_value_quarterly

  [eps] dist=0.596
  Field: eps
Dataset: fundamental6
Description: Earnings Per Share (Basic) - Including Extraordinary Items
Semantics: {"category": "Unknown", "meaning":

  [anl4_af_eps_value] dist=0.602
  Field: anl4_af_eps_value
Dataset: analyst4
Description: Earnings Per Share - Actual Value
Semantics: {"category": "Unknown", "meaning": "anl4_af_eps_v



In [6]:
# Try another semantic query
query2 = "short-term price momentum and volume"
results2 = store.query(COLLECTION_FIELDS, query2, k=5)

print(f"Top 5 fields for query: '{query2}'\n")
for r in results2:
    print(f"  [{r['metadata'].get('field_id')}] dist={r['distance']:.3f}")
    print(f"  {r['document'][:150]}\n")

Top 5 fields for query: 'short-term price momentum and volume'

  [volume] dist=0.576
  volume

  [equity] dist=0.663
  equity

  [assets] dist=0.694
  assets

  [assets_curr] dist=0.761
  assets_curr

  [actual_sales_value_quarterly] dist=0.764
  actual_sales_value_quarterly



## 4. Initialize AlphaMemory (DuckDB)

Creates the database and tables. Will be empty on first run.

In [7]:
from alpha_agent.knowledge.alpha_memory import AlphaMemory

memory = AlphaMemory()
stats = memory.stats()
print("Alpha memory initialized")
print(f"  Total alphas: {stats['total']}")
print(f"  Qualified:    {stats['qualified']}")
print(f"  Pass rate:    {stats['pass_rate']}")

Alpha memory initialized
  Total alphas: 0
  Qualified:    0
  Pass rate:    0.0


## ✅ Notebook 01 Complete

You now have:
- Data fields indexed in Chroma with LLM-generated semantics
- AlphaMemory (DuckDB) initialized and ready

**Next**: Run `02_build_operator_and_paper_kb.ipynb` to index operators and research papers.